In [1]:
import h5py
import numpy as np
import joblib
import torch

In [2]:
print(torch.cuda.is_available())

True


In [5]:
DS_ROOT = '/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_hml3d_0.05'


In [6]:
meta_data = joblib.load(f'{DS_ROOT}/phc_act_amass_train_upright_hml3d_metadata.pkl')
failed_keys = joblib.load(f'{DS_ROOT}/failed.pkl')

In [7]:
print(meta_data.keys())

motion_lengths = np.concatenate( [ml for ml in meta_data['motion_lengths']])
keys_names = np.concatenate( [kn for kn in meta_data['key_names']])

print(len(motion_lengths))
print(sum(motion_lengths))
print(len(keys_names))
print(keys_names)
num_motions = len(keys_names)
motion_starts = np.cumsum(motion_lengths)
motion_starts = np.insert(motion_starts, 0, 0)
motion_starts = motion_starts[:-1]
print(motion_starts[:5])

dict_keys(['key_names', 'motion_lengths', 'running_mean', 'config'])
40960
7737405
40960
['0-KIT_9_bend_right08_poses_sample_0'
 '0-KIT_9_bend_left09_poses_sample_0'
 '0-KIT_291_push_recovery_stand_front06_poses_sample_0' ...
 '0-KIT_674_wash_front01_poses_sample_0'
 '0-KIT_674_wash_front01_poses_sample_1'
 '0-KIT_674_wash_front02_poses_sample_0']
[  0 149 298 447 596]


In [8]:
print(len(failed_keys))
print(failed_keys)

115
['0-BMLmovi_Subject_37_F_MoSh_Subject_37_F_8_poses_sample_0'
 '0-BMLhandball_S07_Expert_Trial_upper_left_right_219_poses_sample_0'
 '0-CMU_114_114_16_poses_sample_0' '0-CMU_114_114_02_poses_sample_0'
 '0-CMU_01_01_01_poses_sample_0' '0-CMU_142_142_21_poses_sample_0'
 '0-CMU_56_56_03_poses_sample_1' '0-CMU_144_144_01_poses_sample_0'
 '0-CMU_144_144_01_poses_sample_1' '0-CMU_77_77_18_poses_sample_0'
 '0-CMU_75_75_07_poses_sample_0' '0-CMU_75_75_08_poses_sample_0'
 '0-CMU_75_75_08_poses_sample_1' '0-CMU_113_113_08_poses_sample_0'
 '0-CMU_111_111_08_poses_sample_0' '0-CMU_135_135_11_poses_sample_0'
 '0-CMU_85_85_12_poses_sample_1' '0-CMU_85_85_13_poses_sample_0'
 '0-CMU_85_85_04_poses_sample_0' '0-CMU_85_85_04_poses_sample_1'
 '0-CMU_13_13_17_poses_sample_1' '0-CMU_90_90_30_poses_sample_0'
 '0-CMU_49_49_03_poses_sample_0'
 '0-TotalCapture_s4_freestyle1_poses_sample_0'
 '0-MPI_Limits_03099_op4_poses_sample_0'
 '0-MPI_Limits_03099_op3_poses_sample_0'
 '0-MPI_Limits_03099_op3_poses_sample

In [9]:
dataset_path = f'{DS_ROOT}/phc_act_amass_train_upright_hml3d.h5'
with h5py.File(dataset_path, 'r') as hdf5_file:
    # Get the total size of the dataset
    for key in hdf5_file.keys():
        print(f"- {key}")
    dataset_size = len(hdf5_file['clean_action']) 

    assert dataset_size == sum(motion_lengths)
    # Load the `reset` boolean array
    reset = hdf5_file['reset'][:]
    action  =  hdf5_file['clean_action'][:]
    obs  =  hdf5_file['pdp_obs'][:]



- clean_action
- pdp_obs
- reset


In [10]:

exclude_ids = []
exclude_indicies = []

obs_dim = obs.shape[-1]
action_dim = action.shape[-1]

count_obs = 0
mean_obs = np.zeros(obs_dim, dtype=obs.dtype)
M2_obs = np.zeros(obs_dim, dtype=obs.dtype)
# Initialize min and max as before
obs_min = np.full(obs_dim, np.inf, dtype=obs.dtype)
obs_max = np.full(obs_dim, -np.inf, dtype=obs.dtype)


count_action = 0
mean_action = np.zeros(action_dim, dtype=obs.dtype)
M2_action = np.zeros(action_dim, dtype=obs.dtype)
# Initialize min and max as before
action_min = np.full(action_dim, np.inf, dtype=obs.dtype)
action_max = np.full(action_dim, -np.inf, dtype=obs.dtype)

for m_i in range(num_motions):

    ml = motion_lengths[m_i]
    start_idx, end_idx = motion_starts[m_i], motion_starts[m_i]+ml


    if keys_names[m_i] in failed_keys:
        exclude_ids.append(m_i)
        exclude_indicies.extend(range(start_idx, end_idx))
        continue
    

    m_r = reset[start_idx:end_idx]
    m_a = action[start_idx:end_idx]
    m_o = obs[start_idx:end_idx]

    has_no_reset = (np.sum(m_r) == 0)
    if has_no_reset:
        exclude_ids.append(m_i)
        exclude_indicies.extend(range(start_idx, end_idx))
        continue

    is_last_a_reset = (m_r[-1] == 1)
    assert is_last_a_reset, 'Last index is not Reset'

    has_only_one_reset = (np.sum(m_r) == 1)
    assert has_only_one_reset, 'More the one Reset in data'

    #OBS -------------- Update min/max
    obs_min = np.minimum(obs_min, m_o.min(axis=0))
    obs_max = np.maximum(obs_max, m_o.max(axis=0))

    # Update mean and M2 using Welford's algorithm -------------
    chunk_count = m_o.shape[0]
    
    delta = m_o - mean_obs
    new_mean_obs = mean_obs + np.sum(delta, axis=0) / (count_obs + chunk_count)
    
    delta2 = m_o - new_mean_obs
    M2_obs += np.sum(delta * delta2, axis=0)
    
    mean_obs = new_mean_obs
    count_obs += chunk_count


    #OBS -------------- Update min/max ----------------
    action_min = np.minimum(action_min, m_a.min(axis=0))
    action_max = np.maximum(action_max, m_a.max(axis=0))

    # Update mean and M2 using Welford's algorithm
    chunk_count = m_o.shape[0]
    
    delta = m_a - mean_action
    new_mean_action = mean_action + np.sum(delta, axis=0) / (count_action + chunk_count)
    
    delta2 = m_a - new_mean_action
    M2_action += np.sum(delta * delta2, axis=0)
    
    mean_action = new_mean_action
    count_action += chunk_count



all_indices = np.arange(len(action))
indices_to_keep = np.setdiff1d(all_indices, exclude_indicies)

std_obs = np.sqrt(M2_obs / count_obs)
std_action = np.sqrt(M2_action / count_action)


In [11]:
import sys
import os


module_path = os.path.abspath('/home/mcarroll/Documents/cd-2/VideoMimic/PDP')

# Insert the path at the beginning of the list
sys.path.insert(0, module_path)
from pdp.utils.normalizer import LinearNormalizer

In [12]:
data = {
    'obs': {
        'min':obs_min,
        'max':obs_max,
        'mean':mean_obs,
        'std':std_obs,
    },
    'action': {
        'min':action_min,
        'max':action_max,
        'mean':mean_action,
        'std':std_action,
    },
}

normalizer = LinearNormalizer()
normalizer.fit_implicit(data=data, mode='limits')


{'min': array([-1.12736642e-01, -1.14820294e-01, -1.15013301e-01, -4.80166644e-01,
       -4.86100793e-01, -4.90794659e-01, -8.51187646e-01, -8.80730510e-01,
       -8.88914526e-01, -9.13623273e-01, -9.21202958e-01, -1.00082827e+00,
       -1.12603351e-01, -1.13021590e-01, -1.13101676e-01, -4.94804978e-01,
       -4.94504094e-01, -4.97513711e-01, -8.58990192e-01, -8.68841708e-01,
       -8.95027518e-01, -9.53044355e-01, -9.97430563e-01, -1.01716292e+00,
       -1.11865021e-01, -1.09027803e-01, -1.11113325e-01, -2.45063663e-01,
       -2.43411914e-01, -2.45265782e-01, -2.97653735e-01, -2.99252659e-01,
       -3.03386450e-01, -5.10474205e-01, -5.07850826e-01, -5.01619935e-01,
       -5.65064132e-01, -5.67557514e-01, -5.82354486e-01, -4.25175130e-01,
       -4.21582937e-01, -4.12700891e-01, -4.68843192e-01, -5.08645594e-01,
       -4.88132775e-01, -5.12491643e-01, -6.68462336e-01, -7.41153181e-01,
       -7.14294791e-01, -8.50488186e-01, -9.86629128e-01, -7.80233979e-01,
       -9.1810506

In [13]:
normalizer_state = normalizer.state_dict()

# Save the state dictionary to a file
torch.save(normalizer_state, f'{DS_ROOT}/normalizer_params.pt')

In [14]:
meta_data['exclude_ids'] = exclude_ids
joblib.dump(meta_data, f'{DS_ROOT}/phc_act_amass_train_upright_hml3d_metadata.pkl')

['/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_hml3d_0.05/phc_act_amass_train_upright_hml3d_metadata.pkl']

# Pre-process clip emb

In [14]:
from pdp.lora_model import load_clip

clip_fn, clip_dim = load_clip(device='cuda')

In [16]:
labels = joblib.load(f'/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_hml3d_0.05/hml3d_labels.pkl')
embs = {}

for k,v  in labels.items():
    clip_emb = clip_fn(v)
    embs[k] = clip_emb.cpu().numpy()


joblib.dump(embs, f'/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_hml3d_0.05/hml3d_embs.pkl')


['/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_hml3d_0.05/hml3d_embs.pkl']